In [ ]:
from pathlib import Path
import json
import shutil

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
BASE_DIR = Path.cwd().parent.resolve()  # Ejecutar desde pec3/notebooks

DATA_DIR = BASE_DIR / "data"
COLAB_DIR = DATA_DIR / "imgsMIIA_colab"

EXTRACTED_DIR = COLAB_DIR / "extracted"
PREPARED_DIR = COLAB_DIR / "prepared"

PREP_IMAGES_DIR = PREPARED_DIR / "images"
PREP_MASKS_DIR = PREPARED_DIR / "masks"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"

for carpeta in [PREP_IMAGES_DIR, PREP_MASKS_DIR, TABLES_DIR, FIGURES_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

print("EXTRACTED_DIR existe:", EXTRACTED_DIR.exists())
print("PREPARED_DIR:", PREPARED_DIR)

In [ ]:
if PREP_IMAGES_DIR.exists():
    shutil.rmtree(PREP_IMAGES_DIR)

if PREP_MASKS_DIR.exists():
    shutil.rmtree(PREP_MASKS_DIR)

PREP_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
PREP_MASKS_DIR.mkdir(parents=True, exist_ok=True)

print("Carpetas preparadas limpias.")

In [ ]:
def detectar_clase_por_texto(texto):
    texto = str(texto).upper()
    
    if "_DT" in texto or "- DT" in texto or " DT" in texto or "DORSAL" in texto:
        return "DT"
    
    if "_VT" in texto or "- VT" in texto or " VT" in texto or "VENTRAL" in texto:
        return "VT"
    
    if "_LT" in texto or "- LT" in texto or " LT" in texto or "LATERAL" in texto:
        return "LT"
    
    return "desconocida"

In [ ]:
def leer_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    return data

In [ ]:
def es_json_via_project(json_path):
    try:
        data = leer_json(json_path)
    except:
        return False
    
    if isinstance(data, dict) and "_via_img_metadata" in data:
        return True
    
    return False

In [ ]:
json_todos = list(EXTRACTED_DIR.rglob("*.json"))

json_utiles = []

for jp in json_todos:
    ruta = str(jp)
    ruta_lower = ruta.lower()
    
    # Ignorar basura de macOS
    if "__macosx" in ruta_lower:
        continue
    
    # Ignorar checkpoints de Jupyter
    if ".ipynb_checkpoints" in ruta_lower:
        continue
    
    # Ignorar los JSON alternativos de Zarza fuera de Proyects
    if "imgsmIIA_Zarza".lower() in ruta_lower:
        if "proyects" not in ruta_lower:
            continue
    
    # Usar solo VIA project
    if not es_json_via_project(jp):
        continue
    
    clase = detectar_clase_por_texto(jp.name)
    
    if clase == "desconocida":
        clase = detectar_clase_por_texto(str(jp))
    
    if clase == "desconocida":
        continue
    
    json_utiles.append({
        "json_path": jp,
        "json_relativo": str(jp.relative_to(EXTRACTED_DIR)),
        "clase_lengua": clase
    })

df_json_utiles = pd.DataFrame(json_utiles)

print("JSON útiles:", len(df_json_utiles))
display(df_json_utiles)

print("\nDistribución por clase:")
print(df_json_utiles["clase_lengua"].value_counts())

In [ ]:
def extraer_anotaciones_via(data, json_path, clase_lengua):
    """
    Extrae anotaciones desde un JSON VIA project.
    """
    
    resultados = []
    
    if "_via_img_metadata" not in data:
        return resultados
    
    metadata = data["_via_img_metadata"]
    
    for img_id in metadata:
        item = metadata[img_id]
        
        filename = item.get("filename", None)
        regions = item.get("regions", [])
        
        if filename is None:
            continue
        
        if isinstance(regions, dict):
            regions = list(regions.values())
        
        resultados.append({
            "json_path": json_path,
            "filename": filename,
            "regions": regions,
            "clase_lengua": clase_lengua
        })
    
    return resultados

In [ ]:
def limpiar_regions(regions):
    if regions is None:
        return []
    
    if isinstance(regions, dict):
        regions = list(regions.values())
    
    if not isinstance(regions, list):
        return []
    
    regions_limpias = []
    
    for region in regions:
        if region is None:
            continue
        
        if not isinstance(region, dict):
            continue
        
        if "shape_attributes" not in region:
            continue
        
        regions_limpias.append(region)
    
    return regions_limpias

In [ ]:
def contar_regiones_utiles(regions):
    regions = limpiar_regions(regions)
    
    total = 0
    
    for region in regions:
        shape = region.get("shape_attributes", {})
        nombre = shape.get("name", "")
        
        if nombre in ["polygon", "rect", "circle", "ellipse"]:
            total += 1
    
    return total

In [ ]:
def raiz_aportacion(json_path):
    """
    Devuelve la carpeta principal de la aportación.
    Ejemplo:
    extracted/imgsMIIA_ffortes/...
    """
    
    rel = json_path.relative_to(EXTRACTED_DIR)
    primera = rel.parts[0]
    
    return EXTRACTED_DIR / primera

In [ ]:
def buscar_imagen_para_json(json_path, filename, clase_lengua):
    """
    Busca la imagen asociada al JSON.
    Primero busca dentro de la aportación concreta.
    Prioriza rutas que contengan la clase DT/VT/LT.
    """
    
    filename = Path(str(filename)).name
    root = raiz_aportacion(json_path)
    
    extensiones = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]
    
    candidatos = []
    
    for p in root.rglob(filename):
        if p.suffix in extensiones:
            if "__MACOSX" not in str(p):
                candidatos.append(p)
    
    # Si no encuentra por nombre exacto, prueba por stem
    if len(candidatos) == 0:
        stem = Path(filename).stem
        
        for ext in extensiones:
            for p in root.rglob(stem + ext):
                if "__MACOSX" not in str(p):
                    candidatos.append(p)
    
    if len(candidatos) == 0:
        return None
    
    # Priorizar candidato cuya ruta indique la clase correcta
    for p in candidatos:
        clase_cand = detectar_clase_por_texto(str(p))
        
        if clase_cand == clase_lengua:
            return p
    
    return candidatos[0]

In [ ]:
def crear_mascara_desde_regions(regions, alto, ancho):
    """
    Fondo = 0
    Región anotada = 255
    """
    
    mask = np.zeros((alto, ancho), dtype=np.uint8)
    
    regions = limpiar_regions(regions)
    
    for region in regions:
        shape = region.get("shape_attributes", {})
        nombre = shape.get("name", "")
        
        if nombre == "polygon":
            xs = shape.get("all_points_x", [])
            ys = shape.get("all_points_y", [])
            
            if len(xs) >= 3 and len(xs) == len(ys):
                puntos = np.array(list(zip(xs, ys)), dtype=np.int32)
                
                puntos[:, 0] = np.clip(puntos[:, 0], 0, ancho - 1)
                puntos[:, 1] = np.clip(puntos[:, 1], 0, alto - 1)
                
                puntos = puntos.reshape((-1, 1, 2))
                cv2.fillPoly(mask, [puntos], 255)
        
        elif nombre == "rect":
            x = int(shape.get("x", 0))
            y = int(shape.get("y", 0))
            w = int(shape.get("width", 0))
            h = int(shape.get("height", 0))
            
            x1 = max(0, x)
            y1 = max(0, y)
            x2 = min(ancho - 1, x + w)
            y2 = min(alto - 1, y + h)
            
            if x2 > x1 and y2 > y1:
                cv2.rectangle(mask, (x1, y1), (x2, y2), 255, -1)
        
        elif nombre == "circle":
            cx = int(shape.get("cx", 0))
            cy = int(shape.get("cy", 0))
            r = int(shape.get("r", 0))
            
            if r > 0:
                cv2.circle(mask, (cx, cy), r, 255, -1)
        
        elif nombre == "ellipse":
            cx = int(shape.get("cx", 0))
            cy = int(shape.get("cy", 0))
            rx = int(shape.get("rx", 0))
            ry = int(shape.get("ry", 0))
            
            if rx > 0 and ry > 0:
                cv2.ellipse(mask, (cx, cy), (rx, ry), 0, 0, 360, 255, -1)
    
    return mask

In [ ]:
filas = []

contador = 0
errores = []

for _, fila_json in df_json_utiles.iterrows():
    json_path = Path(fila_json["json_path"])
    clase_lengua = fila_json["clase_lengua"]
    
    data = leer_json(json_path)
    anotaciones = extraer_anotaciones_via(data, json_path, clase_lengua)
    
    for a in anotaciones:
        n_regiones = contar_regiones_utiles(a["regions"])
        
        if n_regiones == 0:
            continue
        
        img_path = buscar_imagen_para_json(
            json_path,
            a["filename"],
            clase_lengua
        )
        
        if img_path is None:
            errores.append({
                "json": str(json_path),
                "filename": a["filename"],
                "motivo": "imagen_no_encontrada"
            })
            continue
        
        img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        
        if img is None:
            errores.append({
                "json": str(json_path),
                "filename": a["filename"],
                "motivo": "opencv_no_lee_imagen"
            })
            continue
        
        alto, ancho = img.shape[:2]
        
        mask = crear_mascara_desde_regions(
            a["regions"],
            alto,
            ancho
        )
        
        if mask.max() == 0:
            errores.append({
                "json": str(json_path),
                "filename": a["filename"],
                "motivo": "mascara_vacia"
            })
            continue
        
        nombre_aporte = raiz_aportacion(json_path).name
        stem = Path(a["filename"]).stem
        
        nombre_salida = f"{contador:05d}_{nombre_aporte}_{clase_lengua}_{stem}"
        
        img_out = PREP_IMAGES_DIR / f"{nombre_salida}.jpg"
        mask_out = PREP_MASKS_DIR / f"{nombre_salida}.png"
        
        cv2.imwrite(str(img_out), img)
        cv2.imwrite(str(mask_out), mask)
        
        filas.append({
            "id": contador,
            "origen": "imgsMIIA_colab",
            "aportacion": nombre_aporte,
            "clase_lengua": clase_lengua,
            "json_path": str(json_path),
            "json_relativo": str(json_path.relative_to(EXTRACTED_DIR)),
            "filename_original": a["filename"],
            "img_original": str(img_path),
            "img_preparada": str(img_out),
            "mask_preparada": str(mask_out),
            "alto": alto,
            "ancho": ancho,
            "num_regions_utiles": n_regiones
        })
        
        contador += 1

df_masks = pd.DataFrame(filas)
df_errores = pd.DataFrame(errores)

print("Pares imagen/máscara creados:", len(df_masks))
print("Errores:", len(df_errores))

display(df_masks.head())

if len(df_errores) > 0:
    display(df_errores.head())

In [ ]:
csv_masks = TABLES_DIR / "imgsMIIA_masks_validas.csv"
csv_errores = TABLES_DIR / "imgsMIIA_masks_errores.csv"

df_masks.to_csv(csv_masks, index=False)
df_errores.to_csv(csv_errores, index=False)

print("CSV máscaras:", csv_masks)
print("CSV errores:", csv_errores)

In [ ]:
print("Resumen por clase:")
display(df_masks["clase_lengua"].value_counts())

print("Resumen por aportación:")
display(df_masks["aportacion"].value_counts())

print("Resumen cruzado:")
display(pd.crosstab(df_masks["aportacion"], df_masks["clase_lengua"]))

In [ ]:
muestras = df_masks.sample(
    n=min(8, len(df_masks)),
    random_state=7
)

plt.figure(figsize=(12, 8))

for i, (_, fila) in enumerate(muestras.iterrows()):
    img = cv2.imread(fila["img_preparada"], cv2.IMREAD_COLOR)
    mask = cv2.imread(fila["mask_preparada"], cv2.IMREAD_GRAYSCALE)
    
    if img is None or mask is None:
        continue
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.subplot(2, 4, i + 1)
    plt.imshow(img_rgb)
    plt.imshow(mask, cmap="jet", alpha=0.35)
    plt.title(fila["clase_lengua"])
    plt.axis("off")

plt.tight_layout()

fig_path = FIGURES_DIR / "imgsMIIA_mascaras_preparadas.png"
plt.savefig(fig_path, dpi=150)
plt.show()

print("Figura guardada:", fig_path)

In [ ]:
n_imgs = len(list(PREP_IMAGES_DIR.glob("*.jpg")))
n_masks = len(list(PREP_MASKS_DIR.glob("*.png")))

print("Imágenes preparadas:", n_imgs)
print("Máscaras preparadas:", n_masks)

print("Carpeta imágenes:", PREP_IMAGES_DIR)
print("Carpeta máscaras:", PREP_MASKS_DIR)

In [ ]:
zarza_dt_jsons = []

for jp in EXTRACTED_DIR.rglob("*.json"):
    ruta = str(jp).lower()
    
    if "__macosx" in ruta:
        continue
    
    if "zarza" in ruta and "dt" in ruta:
        zarza_dt_jsons.append(jp)

print("JSON DT de Zarza encontrados:", len(zarza_dt_jsons))

for jp in zarza_dt_jsons:
    print(jp)

In [ ]:
zarza_dt_jsons = []

for jp in EXTRACTED_DIR.rglob("*.json"):
    ruta = str(jp).lower()
    
    if "__macosx" in ruta:
        continue
    
    if ".ipynb_checkpoints" in ruta:
        continue
    
    if "zarza" in ruta and "dt" in ruta:
        zarza_dt_jsons.append(jp)

print("JSON DT reales de Zarza encontrados:", len(zarza_dt_jsons))

for jp in zarza_dt_jsons:
    print(jp)

In [ ]:
def extraer_anotaciones_via_flexible(data, json_path, clase_lengua):
    """
    Lee dos formatos posibles:
    1. VIA project: contiene _via_img_metadata
    2. VIA annotations: diccionario con filename/regions directamente
    """
    
    resultados = []
    
    # Caso 1: VIA project completo
    if isinstance(data, dict) and "_via_img_metadata" in data:
        metadata = data["_via_img_metadata"]
        
        for img_id in metadata:
            item = metadata[img_id]
            
            filename = item.get("filename", None)
            regions = item.get("regions", [])
            
            if filename is None:
                continue
            
            if isinstance(regions, dict):
                regions = list(regions.values())
            
            resultados.append({
                "json_path": json_path,
                "filename": filename,
                "regions": regions,
                "clase_lengua": clase_lengua
            })
    
    # Caso 2: VIA annotations exportado
    elif isinstance(data, dict):
        for clave in data:
            item = data[clave]
            
            if not isinstance(item, dict):
                continue
            
            if "filename" not in item or "regions" not in item:
                continue
            
            filename = item.get("filename", None)
            regions = item.get("regions", [])
            
            if filename is None:
                continue
            
            if isinstance(regions, dict):
                regions = list(regions.values())
            
            resultados.append({
                "json_path": json_path,
                "filename": filename,
                "regions": regions,
                "clase_lengua": clase_lengua
            })
    
    return resultados

In [ ]:
for jp in zarza_dt_jsons:
    data_tmp = leer_json(jp)
    anotaciones_tmp = extraer_anotaciones_via_flexible(data_tmp, jp, "DT")
    
    total = len(anotaciones_tmp)
    con_regiones = 0
    sin_regiones = 0
    
    ejemplos = []
    
    for a in anotaciones_tmp:
        n = contar_regiones_utiles(a["regions"])
        
        if n > 0:
            con_regiones += 1
        else:
            sin_regiones += 1
        
        if len(ejemplos) < 5:
            ejemplos.append({
                "filename": a["filename"],
                "regiones_utiles": n
            })
    
    print("\nJSON:", jp)
    print("Total anotaciones:", total)
    print("Con regiones útiles:", con_regiones)
    print("Sin regiones útiles:", sin_regiones)
    display(pd.DataFrame(ejemplos))